In [165]:
# Зареждане на набора от данни
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

In [166]:
# Конструиране на GCN
class GCN(torch.nn.Module):

    def __init__(self,
                 in_channels,
                 hidden_channels,
                 out_channels):

        super().__init__()

        self.conv1 = GCNConv(
            in_channels,
            hidden_channels)

        self.conv2 = GCNConv(
            hidden_channels,
            out_channels)

    def forward(self,
                x,
                edge_index):

        x = self.conv1(
            x,
            edge_index)

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training)

        x = self.conv2(
            x,
            edge_index)

        return x

In [167]:
# Създаване на GCN модела
gcn = GCN(
    dataset.num_features,
    hidden_channels=64,
    out_channels=dataset.num_classes)

In [172]:
# Конструиране на GAT
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
class GAT(torch.nn.Module):

    def __init__(self,
                 in_channels,
                 hidden_channels,
                 out_channels,
                 heads=8):

        super().__init__()

        self.conv1 = GATConv(
            in_channels,
            hidden_channels,
            heads=heads,
            dropout=0.6)

        self.conv2 = GATConv(
            hidden_channels * heads,
            out_channels,
            heads=1,
            concat=False,
            dropout=0.6)

    def forward(self,
                x,
                edge_index):

        x = self.conv1(
            x,
            edge_index)

        x = F.elu(x)

        x = F.dropout(
            x,
            p=0.6,
            training=self.training)

        x = self.conv2(
            x,
            edge_index)

        return x

In [173]:
# Създаване на GAT модела
gat = GAT(
    dataset.num_features,
    hidden_channels=8,
    out_channels=dataset.num_classes,
    heads=8)

In [174]:
# Конструиране на GraphSAGE
class GraphSAGE(torch.nn.Module):

    def __init__(self,
                 in_channels,
                 hidden_channels,
                 out_channels):

        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels)

        self.conv2 = SAGEConv(
            hidden_channels,
            out_channels)

    def forward(self,
                x,
                edge_index):

        x = self.conv1(
            x,
            edge_index)

        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training)

        x = self.conv2(
            x,
            edge_index)

        return x

In [175]:
# Създаване на GraphSAGE модела
graphsage = GraphSAGE(
    dataset.num_features,
    hidden_channels=64,
    out_channels=dataset.num_classes)

In [176]:
# Дефиниране на енкодера
class Encoder(torch.nn.Module):

    def __init__(self,
                 in_channels,
                 hidden_channels):

        super().__init__()

        self.conv = GCNConv(
            in_channels,
            hidden_channels)

    def forward(self,
                x,
                edge_index):

        x = self.conv(
            x,
            edge_index)

        return x.relu()

In [177]:
# Създаване на DeepGraphInfomax модела
dgi = DeepGraphInfomax(
    hidden_channels=64,
    encoder=Encoder(
        dataset.num_features,
        64),
    summary=lambda z,
                    *args,
                    **kwargs:
        torch.sigmoid(
            z.mean(dim=0)),

    corruption=lambda x,
                       edge_index:
        (
            x[
                torch.randperm(
                    x.size(0)
                )
            ],
            edge_index
        )
)

In [178]:
# Обучение на GCN, GAT и GraphSAGE
import time
import torch.nn.functional as F

def train_supervised(model, data, epochs=200):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.01,
        weight_decay=5e-4)

    start = time.time()

    model.train()

    for epoch in range(epochs):

        optimizer.zero_grad()

        out = model(
            data.x,
            data.edge_index)

        loss = F.cross_entropy(
            out[data.train_mask],
            data.y[data.train_mask])

        loss.backward()

        optimizer.step()

    train_time = time.time() - start

    return train_time

In [179]:
# Обучение на Deep Graph Infomax
from sklearn.linear_model import LogisticRegression

def train_dgi(model, data, epochs=200):

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001)

    start = time.time()

    model.train()

    for epoch in range(epochs):

        optimizer.zero_grad()

        pos_z, neg_z, summary = model(
            data.x,
            data.edge_index)

        loss = model.loss(
            pos_z,
            neg_z,
            summary)

        loss.backward()

        optimizer.step()

    train_time = time.time() - start

    model.eval()

    with torch.no_grad():

        embeddings = model.encoder(
            data.x,
            data.edge_index)

    clf = LogisticRegression(
        max_iter=1000)

    clf.fit(
        embeddings[data.train_mask].cpu(),
        data.y[data.train_mask].cpu())

    return train_time, embeddings, clf

In [180]:
# Последователно обучение на моделите
gcn_time = train_supervised(gcn, data)

gat_time = train_supervised(gat, data)

graphsage_time = train_supervised(graphsage, data)

dgi_time, embeddings, clf = train_dgi(dgi, data)

In [181]:
# Оценяване
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

def evaluate_supervised(model, data):

    model.eval()

    with torch.no_grad():

        out = model(
            data.x,
            data.edge_index)

        pred = out.argmax(dim=1)

    accuracy = accuracy_score(
        data.y[data.test_mask].cpu(),
        pred[data.test_mask].cpu()
    )

    f1 = f1_score(
        data.y[data.test_mask].cpu(),
        pred[data.test_mask].cpu(),
        average="macro"
    )

    return accuracy, f1

In [182]:
# Оценяване на Deep Graph Infomax
def evaluate_dgi(clf, embeddings, data):

    pred = clf.predict(
        embeddings[data.test_mask].cpu()
    )

    accuracy = accuracy_score(
        data.y[data.test_mask].cpu(),
        pred
    )

    f1 = f1_score(
        data.y[data.test_mask].cpu(),
        pred,
        average="macro"
    )

    return accuracy, f1

In [183]:
# Изчисляване на показателите
gcn_acc, gcn_f1 = evaluate_supervised(
    gcn,
    data)

gat_acc, gat_f1 = evaluate_supervised(
    gat,
    data)

sage_acc, sage_f1 = evaluate_supervised(
    graphsage,
    data)

dgi_acc, dgi_f1 = evaluate_dgi(
    clf,
    embeddings,
    data)

In [184]:
# Брой параметри
def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )
gcn_params = count_parameters(gcn)
gat_params = count_parameters(gat)
sage_params = count_parameters(graphsage)
dgi_params = count_parameters(dgi)    

In [185]:
# Отпечатване на резултатите
print(f"{'Model':<15} {'Accuracy':>10} {'Macro F1':>10} {'Parameters':>12} {'Time (s)':>10}")

print("-" * 62)

print(f"{'GCN':<15} {gcn_acc:>10.4f} {gcn_f1:>10.4f} {gcn_params:>12} {gcn_time:>10.2f}")

print(f"{'GAT':<15} {gat_acc:>10.4f} {gat_f1:>10.4f} {gat_params:>12} {gat_time:>10.2f}")

print(f"{'GraphSAGE':<15} {sage_acc:>10.4f} {sage_f1:>10.4f} {sage_params:>12} {graphsage_time:>10.2f}")

print(f"{'DGI':<15} {dgi_acc:>10.4f} {dgi_f1:>10.4f} {dgi_params:>12} {dgi_time:>10.2f}")

Model             Accuracy   Macro F1   Parameters   Time (s)
--------------------------------------------------------------
GCN                 0.8090     0.8019        92231       3.81
GAT                 0.8060     0.7987        92373       7.33
GraphSAGE           0.8050     0.7969       184391       9.90
DGI                 0.7950     0.7811        95872       5.73
